# Previsão de chuva no Rio Grande do Sul (NASA POWER + ML)

**Como usar:** execute as células em ordem (Runtime > Run all).

**Sobre os dados:** a API NASA POWER não é "tempo real". Os dados diários chegam com alguns dias de atraso (em geral 2 a 7 dias). Por isso o modelo prevê "chuva no dia seguinte ao último dia disponível", e não a previsão do tempo de amanhã de verdade.

**Variáveis usadas:** precipitação, temperatura (média, máx., mín.), umidade relativa, vento e pressão de superfície (assumi que "pilares" era pressão).

In [ ]:
import os
import numpy as np
import pandas as pd
import requests
import time
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, f1_score, mean_absolute_error

# Cidades de referência no RS (nome: (latitude, longitude))
CIDADES = {
    "Porto Alegre": (-30.03, -51.23),
    "Caxias do Sul": (-29.17, -51.18),
    "Pelotas": (-31.77, -52.34),
    "Santa Maria": (-29.69, -53.81),
    "Passo Fundo": (-28.26, -52.41),
    "Uruguaiana": (-29.76, -57.09),
    "Santa Rosa": (-27.87, -54.48),
    "Bagé": (-31.33, -54.10),
    "Erechim": (-27.63, -52.27),
    "Santa Cruz do Sul": (-29.72, -52.43),
    "Lajeado": (-29.47, -51.96),
    "Vacaria": (-28.51, -50.93),
    "Rio Grande": (-32.03, -52.10),
    "São Borja": (-28.66, -56.00),
}

# Retângulo aproximado do RS (para validar pesquisas por coordenada)
LIMITES_RS = dict(lat_min=-33.75, lat_max=-27.08, lon_min=-57.65, lon_max=-49.69)
INICIO = "20150101"

## 1. Baixar os dados da NASA POWER

In [ ]:
RENOMEAR = {
    "PRECTOTCORR": "chuva",   # precipitação (mm/dia)
    "T2M": "temp",            # temperatura média a 2 m (°C)
    "T2M_MAX": "tmax",
    "T2M_MIN": "tmin",
    "RH2M": "umid",           # umidade relativa (%)
    "WS2M": "vento",          # vento a 2 m (m/s)
    "PS": "pressao",          # pressão de superfície (kPa)
}

def baixar_power(lat, lon, inicio=INICIO):
    fim = (pd.Timestamp.today() - pd.Timedelta(days=3)).strftime("%Y%m%d")
    r = requests.get(
        "https://power.larc.nasa.gov/api/temporal/daily/point",
        params=dict(
            parameters=",".join(RENOMEAR), community="AG",
            latitude=lat, longitude=lon, start=inicio, end=fim, format="JSON",
        ),
        timeout=180,
    )
    r.raise_for_status()
    dados = r.json()["properties"]["parameter"]
    df = pd.DataFrame(dados).rename(columns=RENOMEAR)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "data"
    df = df.replace(-999, np.nan)
    df = df.loc[:df["chuva"].last_valid_index()]  # corta dias finais ainda sem dados
    return df

ARQ = "rs_power.csv"
if os.path.exists(ARQ):
    dados = pd.read_csv(ARQ, parse_dates=["data"], index_col="data")
else:
    partes = []
    for nome, (lat, lon) in CIDADES.items():
        print("Baixando", nome, "...")
        d = baixar_power(lat, lon)
        d["cidade"], d["lat"], d["lon"] = nome, lat, lon
        partes.append(d)
        time.sleep(1)
    dados = pd.concat(partes)
    dados.to_csv(ARQ)   # cache; apague o arquivo para baixar de novo

print(dados.shape, "| último dia com dados:", dados.index.max().date())
dados.head()

## 2. Criar as variáveis do modelo

Usa os últimos 3 dias de cada variável, chuva acumulada em 7 e 30 dias, variação de pressão e época do ano. O alvo é a chuva do **dia seguinte** (chove = pelo menos 1 mm).

In [ ]:
BASE = ["chuva", "temp", "tmax", "tmin", "umid", "vento", "pressao"]

def criar_features(df):
    df = df.sort_index().copy()
    for c in BASE:
        for l in (1, 2, 3):
            df[f"{c}_l{l}"] = df[c].shift(l)
    df["chuva_7d"] = df["chuva"].rolling(7).sum()
    df["chuva_30d"] = df["chuva"].rolling(30).sum()
    df["dif_pressao"] = df["pressao"].diff()
    df["dif_temp"] = df["temp"].diff()
    doy = df.index.dayofyear
    df["sen_ano"] = np.sin(2 * np.pi * doy / 365.25)
    df["cos_ano"] = np.cos(2 * np.pi * doy / 365.25)
    df["alvo_mm"] = df["chuva"].shift(-1)
    df["alvo_chuva"] = (df["alvo_mm"] >= 1).astype(float).where(df["alvo_mm"].notna())
    return df

partes = []
for cidade, g in dados.groupby("cidade"):
    f = criar_features(g.drop(columns=["cidade"]))
    f["cidade"] = cidade
    partes.append(f)
feats = pd.concat(partes)

FEATURES = [c for c in feats.columns if c not in ("alvo_mm", "alvo_chuva", "cidade")]
print(len(FEATURES), "variáveis")

## 3. Treinar e avaliar

Divisão por tempo: treino até 2023, teste de 2024 em diante (nunca embaralhar séries temporais). Comparo com um modelo ingênuo: "vai chover amanhã se choveu hoje".

In [ ]:
com_alvo = feats.dropna(subset=["alvo_mm"])
treino = com_alvo[com_alvo.index < "2024-01-01"]
teste = com_alvo[com_alvo.index >= "2024-01-01"]

clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=42)
clf.fit(treino[FEATURES], treino["alvo_chuva"])

reg = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
reg.fit(treino[FEATURES], treino["alvo_mm"])

prob = clf.predict_proba(teste[FEATURES])[:, 1]
mm_prev = np.clip(reg.predict(teste[FEATURES]), 0, None)
ingenuo = (teste["chuva"] >= 1).astype(int)

print("Chove amanhã? (classificação)")
print("  ROC-AUC do modelo :", round(roc_auc_score(teste["alvo_chuva"], prob), 3))
print("  F1 do modelo      :", round(f1_score(teste["alvo_chuva"], prob >= 0.5), 3))
print("  F1 do ingênuo     :", round(f1_score(teste["alvo_chuva"], ingenuo), 3))
print("Quanto chove amanhã? (mm)")
print("  MAE do modelo     :", round(mean_absolute_error(teste["alvo_mm"], mm_prev), 2), "mm")
print("  MAE do ingênuo    :", round(mean_absolute_error(teste["alvo_mm"], teste["chuva"]), 2), "mm")

## 4. Gráficos de exemplo (período de teste)

In [ ]:
cid = "Porto Alegre"
t = teste[teste["cidade"] == cid].copy()
t["prob"] = clf.predict_proba(t[FEATURES])[:, 1]
t["mm_prev"] = np.clip(reg.predict(t[FEATURES]), 0, None)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=t.index, y=t["alvo_mm"], name="Chuva real do dia seguinte (mm)", secondary_y=False)
fig.add_scatter(x=t.index, y=t["mm_prev"], name="Previsto (mm)", mode="lines", secondary_y=False)
fig.add_scatter(x=t.index, y=t["prob"], name="Probabilidade de chuva", mode="lines", secondary_y=True)
fig.update_yaxes(title_text="mm", secondary_y=False)
fig.update_yaxes(title_text="probabilidade", range=[0, 1], secondary_y=True)
fig.update_layout(title=f"{cid}: real x previsto (2024 em diante)", height=450)
fig.show()

mensal = feats.groupby([feats.index.month, "cidade"])["chuva"].mean().unstack().mul(30)
px_fig = go.Figure([go.Scatter(x=mensal.index, y=mensal[c], name=c, mode="lines") for c in mensal.columns])
px_fig.update_layout(title="Chuva média mensal por cidade (mm/mês, 2015 até hoje)", xaxis_title="mês", height=450)
px_fig.show()

## 5. Interface: mapa do RS + pesquisa

- O mapa mostra a probabilidade de chuva para cada cidade de referência (verde baixa, laranja média, vermelha alta).
- Escolha uma cidade ou digite latitude/longitude de qualquer ponto do RS e clique em **Analisar**.

In [ ]:
def cor(p):
    return "green" if p < 0.3 else ("orange" if p < 0.6 else "red")

def prever_ultimo_dia(f):
    ult = f.tail(1)
    p = float(clf.predict_proba(ult[FEATURES])[0, 1])
    mm = float(max(reg.predict(ult[FEATURES])[0], 0))
    return p, mm

ultimo = feats.groupby("cidade").tail(1).copy()
ultimo["prob"] = clf.predict_proba(ultimo[FEATURES])[:, 1]
ultimo["mm"] = np.clip(reg.predict(ultimo[FEATURES]), 0, None)

def montar_mapa(ponto=None):
    # Mapa estático (matplotlib): sem tiles/internet, então não sofre os
    # bloqueios que o Colab impõe a iframes e a servidores de mapa (OSM).
    fig, ax = plt.subplots(figsize=(6, 7))
    for _, r in ultimo.iterrows():
        ax.scatter(r["lon"], r["lat"], s=220, c=cor(r["prob"]), edgecolors="black", zorder=3)
        ax.annotate(r["cidade"], (r["lon"], r["lat"]), xytext=(6, 4),
                    textcoords="offset points", fontsize=8)
    if ponto:
        lat, lon, p, mm = ponto
        ax.scatter(lon, lat, s=280, marker="*", c="royalblue", edgecolors="black",
                   zorder=4, label=f"Ponto pesquisado: {p*100:.0f}%, ~{mm:.1f} mm")
        ax.legend(loc="lower left", fontsize=8)
    ax.set_xlim(LIMITES_RS["lon_min"], LIMITES_RS["lon_max"])
    ax.set_ylim(LIMITES_RS["lat_min"], LIMITES_RS["lat_max"])
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Chance de chuva no RS (dia seguinte)")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    plt.show()

def grafico_serie(df, titulo):
    d = df.tail(365)
    fig = make_subplots(
        rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.04,
        subplot_titles=("Chuva (mm/dia)", "Temperatura (°C)", "Umidade relativa (%)", "Vento (m/s)", "Pressão (kPa)"),
    )
    fig.add_bar(x=d.index, y=d["chuva"], row=1, col=1)
    fig.add_scatter(x=d.index, y=d["temp"], row=2, col=1, mode="lines")
    fig.add_scatter(x=d.index, y=d["umid"], row=3, col=1, mode="lines")
    fig.add_scatter(x=d.index, y=d["vento"], row=4, col=1, mode="lines")
    fig.add_scatter(x=d.index, y=d["pressao"], row=5, col=1, mode="lines")
    fig.update_layout(height=850, showlegend=False, title=titulo)
    fig.show()

w_cidade = widgets.Dropdown(options=list(CIDADES), description="Cidade:")
w_lat = widgets.FloatText(value=CIDADES[w_cidade.value][0], description="Lat:")
w_lon = widgets.FloatText(value=CIDADES[w_cidade.value][1], description="Lon:")
w_btn = widgets.Button(description="Analisar", button_style="primary")
out = widgets.Output()

def ao_mudar_cidade(change):
    w_lat.value, w_lon.value = CIDADES[change["new"]]
w_cidade.observe(ao_mudar_cidade, names="value")

def analisar(_=None):
    lat, lon = w_lat.value, w_lon.value
    with out:
        clear_output(wait=True)
        L = LIMITES_RS
        if not (L["lat_min"] <= lat <= L["lat_max"] and L["lon_min"] <= lon <= L["lon_max"]):
            print("Coordenada fora do Rio Grande do Sul.")
            return
        print("Buscando dados na NASA...")
        df = baixar_power(lat, lon)
        f = criar_features(df)
        f["lat"], f["lon"] = lat, lon
        p, mm = prever_ultimo_dia(f)
        clear_output(wait=True)
        print(f"Último dia com dados: {df.index.max().date()}")
        print(f"Dia seguinte: {p*100:.0f}% de chance de chuva, cerca de {mm:.1f} mm")
        montar_mapa((lat, lon, p, mm))
        grafico_serie(df, f"Últimos 12 meses ({lat:.2f}, {lon:.2f})")

w_btn.on_click(analisar)
display(widgets.HBox([w_cidade, w_lat, w_lon, w_btn]), out)
analisar()